# Week 01 · Models of Computation
## Interning, homework review, interactive Turing machines, and the halting problem

**Goals:** distinguish equality from identity; explain how a transition table executes; trace the five example machines; distinguish a timeout from a proof of nontermination; explain why a universal halting decider cannot exist.

Suggested pace: interning 15 min → homework 15 min → machines 40–50 min → halting problem 20 min → exit questions 5 min. Allow a short break for a session of about 100–110 minutes. Copying and the final loop-detection experiment can be explored independently if time is short.

Run cells from top to bottom. Before each **Predict** prompt, discuss your answer before executing. The notebook contains worked homework solutions, so use it after submitting Week 00.

**Environment:** Python 3.11+, Jupyter/VS Code or Colab. The engine has no external dependencies. The buttons need `ipywidgets`; a plain step-by-step alternative is included. The notebook uses the repository's engine when available, otherwise an embedded snapshot of the same source and examples. No repository download is needed.


## 1. Interning: equal values, shared objects

**Predict:** if two objects compare equal, must `is` also return `True`?

`==` asks about equality; `is` asks about identity. Immutable values can sometimes share an object safely. Sharing is an implementation choice, so program correctness must not depend on incidental identity of numbers or strings.


In [ ]:
import platform
import sys
print(platform.python_implementation(), sys.version.split()[0])

for text in ['42', '1000']:
    a = int(text)  # Runtime construction, rather than two compiled literals.
    b = int(text)
    print(text, 'equal:', a == b, '; same object:', a is b)

left, right = [1, 2], [1, 2]
print('Lists:', left == right, left is right)


CPython documents a cache of integer objects from **−5 through 256**. Reusing a cached integer is related to string interning but is a separate mechanism. Identical literals can also share compiled constants; therefore observations may differ between a single cell, separate cells, and runtime construction. Do not use `is` for numeric equality. [CPython integer-object documentation](https://docs.python.org/3/c-api/long.html#c.PyLong_FromLong)

For strings, **interning** chooses a canonical object for a value. `sys.intern(s)` looks up the value in an internal table and returns the shared string. Keep the returned reference. Python commonly interns names used in programs; arbitrary equal strings need not already be interned. [Python: sys.intern](https://docs.python.org/3/library/sys.html#sys.intern)


In [ ]:
import sys
a = 'hello_world_of_python'
b = 'hello_world_of_python'
#a = '_'.join(['Hello', 'world', 'of', 'Python'])
#b = '_'.join(['Hello', 'world', 'of', 'Python'])
print('Before:', a == b, a is b)  # Observe; do not assert incidental identity.
a = sys.intern(a)
b = sys.intern(b)
print('After: ', a == b, a is b)
assert a is b


**Why do it?** Repeated strings can share storage. In dictionary lookups, interned keys and an interned lookup string can use an identity comparison after hashing. Interning itself also costs work, so it is not automatically a speedup. Use it selectively for repeatedly used names or tokens, and measure your actual workload. [Python: sys.intern](https://docs.python.org/3/library/sys.html#sys.intern)

Conceptually: `value → lookup in intern table → existing canonical string, or register one`. This is an optimization of representation, not a change to string equality.

**Discuss:** why is sharing immutable strings safe, while sharing mutable lists can change program behavior? What goes wrong with `user_input is "yes"`? Use `user_input == "yes"`; reserve identity checks for questions such as `value is None`.


In [ ]:
# Count distinct objects while keeping every reference alive.
raw = [''.join(['state', '_', str(i % 3)]) for i in range(100)]
canonical = [sys.intern(s) for s in raw]
print('Distinct values:', len(set(raw)))
print('Objects before interning:', len({id(s) for s in raw}))
print('Objects after interning:', len({id(s) for s in canonical}))
assert raw == canonical
# This is an object-count experiment, not a total-memory benchmark.


## 2. Week 00 homework: a quick review

Answers are not given here. Check the recording of the webinar for details.

## 3. A Turing machine we can inspect

A configuration consists of **state + head position + tape contents**. One transition reads a symbol, writes a symbol, moves the head, and changes state. The tape is unbounded in the mathematical model; our simulator stores only nonblank cells and has practical execution limits.

In this repository the rule format is:

```text
STATE READ -> WRITE MOVE NEXT_STATE
q0    0    -> 1     R    q0
```

`_` is blank; `L`, `R`, `S` mean left, right, stay. The head starts at position 0 in `q0`. `HALT`, `ACCEPT`, and `REJECT` are terminal states. A missing transition raises a simulator error; it is not automatically acceptance or proof of an infinite computation. Other textbook conventions may treat a missing rule as halting.

The five `.tm` files are programs. `yandex_checker.py` is a grading adapter, not a sixth transition-table example.

### Setup

Run the next cell once. It imports the existing engine, without replacing its transition semantics. The embedded fallback makes this notebook usable on its own; when the repository is present, its current files take precedence.


In [6]:
import base64
import importlib
import io
from pathlib import Path
import sys
import os
import tempfile
import zipfile

EMBEDDED_EXAMPLES = {'binary_increment.tm': '# Find the right edge of the binary number.\nq0 0 -> 0 R q0\nq0 1 -> 1 R q0\nq0 _ -> _ L q_carry\n\n# Propagate the carry towards the left.\nq_carry 1 -> 0 L q_carry\nq_carry 0 -> 1 S HALT\n# Overflow: add a new leading one.\nq_carry _ -> 1 S HALT\n\n', 'copy.tm': 'q0 0 -> 0 R q0\nq0 1 -> 1 R q0\nq0 _ -> @ L q1\nq1 0 -> 0 L q1\nq1 1 -> 1 L q1\nq1 _ -> _ R q2\nq2 0 -> X R r0\nq2 1 -> X R r1\n\nr0 0 -> 0 R r0\nr0 1 -> 1 R r0\nr0 @ -> @ R r0\nr1 0 -> 0 R r1\nr1 1 -> 1 R r1\nr1 @ -> @ R r1\n\nr0 _ -> 0 L l0\nr1 _ -> 1 L l1\n\nl0 0 -> 0 L l0\nl0 1 -> 1 L l0\nl0 @ -> @ L l0\nl1 0 -> 0 L l1\nl1 1 -> 1 L l1\nl1 @ -> @ L l1\n\nl0 X -> 0 R q2\nl1 X -> 1 R q2\n\nq2 @ -> @ S HALT', 'even_1.tm': 'q0 0 -> 0 R q0\nq1 0 -> 0 R q1\nq0 1 -> 1 R q1\nq1 1 -> 1 R q0\nq0 _ -> E L q2\nq1 _ -> O L q2\n\nq2 0 -> _ L q2\nq2 1 -> _ L q2\nq2 _ -> _ S HALT\n', 'invert_bits.tm': '# Invert every bit while scanning from left to right.\nq0 0 -> 1 R q0\nq0 1 -> 0 R q0\n# Stop on the first blank after the input.\nq0 _ -> _ S HALT\n\n', 'unary_addition.tm': '# Delete the separator, then shift the right operand one cell left.\nq0 1 -> 1 R q0\nq0 + -> _ R q_take\n\n# Pick up the next unary digit.\nq_take 1 -> _ L q_put\nq_take _ -> _ L HALT\n# Put it into the gap and advance to the following source cell.\nq_put _ -> 1 R q_skip\nq_skip _ -> _ R q_take\n\n'}
ENGINE_ZIP = 'UEsDBBQAAAAIANSEL10dbSZ7/QEAAKQEAAAaAAAAdHVyaW5nX21hY2hpbmUvX19pbml0X18ucHl1U0tr20AQvutXDHuSimxyDAYVSqFgSIpx3VMIYi2NoiWrXXV31SaQH9/ZhyzLpTpI855P38wwxg7TWYoGvhz20GkDrkc4TUaoF3jkTS8UAg6T5E6bLWMsyzqjBxi566U4gxhGbRwcSI0O9z761GTfOzT8LDFlbZsem1c0s/urV4/o5RJOaN0R7SRJDnG11XJyQquUPCQ4Kfk4qTn8h8NxrzpdJuQJeMobubFLz4PRL4YPFGq4ssLXLyGE1OPs+s2laLm7WLIsa7EDqXlbJxR5BvR4GnZgnYGPwEEZrJ/i5yy5eo3eCljNolUo6sllbR01uHh/3SV3j9Rj1BHYjqIdue+ib+BvMc1Gxwd818RHFT4xhEup/2Bb2/fhrCXFzRN4okbPtxkFbD6vKduFKjTnB/rXhYgSuGqh0YqqTI0DDvMwAsUcfp6+be4h0QWdkBiWxRdLUKipm0aJ+Q3EAkR3CxuEBaVdBIvSYpBCNYdvnhFPdu7ZL7bGM+bNOapGt/QvFZtct7lnRZxRAlX9M9bcZ5VXtFaLWN5iqma8oahBNxm15i6uxFXL8mIIq1CF92JcbUK10pag1T5UK20J+g/UGFDQ9tY1hdQ1cfAUB3x1eawElo7Ci5ez8sp8WF5e7jNol+tJi8tWXPiQ9RF7y/UBeX11dt5wOyGq/Zz9BVBLAwQUAAAACADUhC9d4nyTCDAAAAAwAAAAGgAAAHR1cmluZ19tYWNoaW5lL19fbWFpbl9fLnB5SyvKz1XQS87JVMjMLcgvKlHITczM4+IqSswsTlUIriwuSc11rcgs0QAJa2hqcgEAUEsDBBQAAAAIANSEL10hJxvnJwQAAJAOAAAZAAAAdHVyaW5nX21hY2hpbmUvY2hlY2tlci5wec1WTY/bNhC9+1cwOkmFarjXBVS0KDboAk1bbPa2WBC0NF4ToUmVpLJ20v73Dj8kUbLsbJBLdNi1Zp5G8/HmUVmW/Q6sEWAM2YNoQRuyU5rUSlowltR7qD+gcZ1l2Wq10+pAKN11ttNAKeGHVmlLmJTKMsuVNBHTMMtqwYwB04MGU0DYU8vlc++8s6DZVkB8fA1aKz08mq8IXnfyIxO8uZNtZ38V7Z5twd46XOnd77gxGPFBM2m4yyXx/a3Vs2aH9ydp2TGx34NRna7hD37g9vZYAzTQJP73FtpLvodO4+vesXrPJUR7EdM/BGuf/wQaIS3TBnSP8He0DWmWxBfK7GBZrVa/DP3LMcAnkNWD7qBYeRN5wFlhMZ2wNz651rW+uSFbpYQ3cNc16mLcEGO1t8Gxhdo6WG9hte2Y8PfkX/Knwmyd2WAbzA3GsP52xyUT1ODEIQnmOnD2pLdSnDWkLlIFxJeq+s2R7x5chy6U5TiKmQlu7OPYgyeM3MCO0BpTfFb6lMfs3Bhbx42C/PizSyiE5TvCkTtYkawhYMsFzhQB7S4NuAGSZDxwsp9TdjncMj0XQh4CkNoBeSXqMkEXoroRUuGQFCL0StjLe7EQWkfw68Nf3OQrHWYRF8L2XjhC3bkeUR86i4P3qkWNEt4X5COOiFo4Wk/GMmVQL0CPtmvxr3M7zNNTQP0Q/h3YkY7LgDT+aUM3m83olEpuBZMfaA1CRNSE82kcJOdlCBNCvUBDzemwVSLN0KW19ASXyJbJYqI7+2eTBbfPazDTaO01gM6Xev4GvzFnC4mnwn0nXbKkZk7sX7jdqw4FDRXPIo2xiwKg9b9UL3dO+D9yg1LDP/ljA4+bBvwR44KGSJXf6tzPpwgyFFqBHj+kfNaiwpFtZkPyETybQikgDIzSZPVpZFskB4aeS28+YIIAJTqdp5QqyglwnHA1/pxCZqlW8f8IKuKEnGYtHDeEmai6ydI4+XMdepy8apTG/C3DLpTJeVAOHCjDqMmmxLl6+oeFLcq5khbFJLz7YFgKiMYwzAH9NN/vhFF9arGIYrVKSjrXeFdkCPfF148N6g/matrP2ZD7Y9ivTOX/ltP1qiZ35aVZzkZ5RjtfIO5P1Se2xrtpMu4ai6uSOs9ggzpVw69l0FSlqnPT9LHpsHHJlkRjsmlMNq6udep/Uy2LzVmGceJr1rYgmzzh7hnUXVcJ7ZJAOUJX+B2aMs/tvEvu2mW3PZE8loQ6Pi9V8Ub/V5JnLP/zLDQ6suXweLLVSmsMlQZaAM+WDQROYKwr7ev328vsr5BsoyDQ5MBsvR83FVW3w9pJ9qIVfvuE17ymF+bbCeS+OL+x5iicw9lyIeWvkfKvLOQVst5LTKxhvF2c2jX1n8lDkX6SpYKOqpiHGtbhy91LdbA4ce6VPtH8/wFQSwMEFAAAAAgA1IQvXXjufrsEBQAAJRAAABUAAAB0dXJpbmdfbWFjaGluZS9jbGkucHm1V1uL4zYUfs+vEH6yi+1NBwol4JZSttCHboedZV+GQSixnIiRZa8kzyTd7n/v0dV2nKRd2s7LxEfn8p2LPklJkvzctS0RdcGZoIgJTWVDdhQ1nURKDzUVWpVJkqxWjexahHEz6EFSjBFr+05qRIToNNGsE2q1CjK574lUNHyrkwo/tQTvW7J7dv56og+cbYOze/iMRodBM+7jlkGDd6TGLdkdAK5folJ2UgWFD4NkYv+b03hrlryaNwp67wfxnqqBa79sActZmF52e0naHL0QzmqiaZB4kxemBlj5I/r8GAQyd4liU9XValXTBmEXIc1Q8UOsUPmT3A8tFPneLm5WCP48lOqaVmpgVIlukxzVVO0k6039qwRyQgQpyJ/TQpOe+mogX44km/gvSV1j4h2nSVHUdDvswSXZOW9Kd9BoLQcKwgPlPYgO3Su6P+lDJ8ZOKu925yZJAfBJBDVs3ZdKAauuEq8GPiX9NDBJ6+oDxMhW1kkoNTgJ/qwbX7wkrEdI0YCgUsNAuQ55SGHxLNegBF069bQyU3dTvyhaciwUjDlVwQi2iocsoepX0MJSBCptc5YYQf5P4S1VmegHDYoCRKpKfrQT0RAY6yrx/tuuNtUMti3sX8L5CdPjjg+KvVC8l93Qp6P2Invb6suTccPKbY9bZst8isJSkLF4uRnxkmlNOTmFqjWwhfVYjXX53XXDVybq7nXS2dHuXSfodUM3FrRXF22/XeP1en3bWnRiy4l4LnaU8/lw3Q66mMXrBWWaEe5MpvPxaX2jlgdK6otJ3cjHDqOJI/XXmtoaTMHhG9i2kpLnmE8YEdL31NJK8PH49PcuaH/JwxJ69EXh9BOe3yKxA8dqLO1pkrp/m/F8sXxvxsiTu1FOm+QXJghHhqQ3IP3szMpu0FDDL4GoZ7o24c2o2xgpttJzgwczk8av/QsGdlKXqkQPUXdUNVKj63OEIqaGYzbjofSOtFT1wAw2Q/DmEgznbDU7q9NVQGO8lPFotY2vrMz+zJGfV5eYW5mJ8ujJzCjuO8Vs+6ymEeUA4eh0lZOO3844mwItIWWqz/DZUf5PsFhP2G6KanTtBFM0oR+/Ov9+Mj6PJqFxrHEQHb0iuKd5ncibUWan3CFdyGi/ieDHe4vtcWmJNHf6jhvzRZCsNCMRXUwKmp8JjzbarBXwudQKVIgtFY7qc/ncbpKhqkwbz/POLusbPAt12k+0s/jLbQmY58nAmJ3uNCgPDbGn5FhU19DxGpiG8lgymFU3uxVsWeX/s6CdsN4qTtptTWB6my5Hu0FKIM7NMie/knvFWU7ntaRcTcrzLxL9iuTOIFyi6hmxrz3ftYRZvnvZIM6UflRaPqE/becAsr0UzEjPhIeFeMd3zwlz5CjrZr51XYUQU5NzwfyZeySnjmjdph8HQ5l3RuVfReWeagxbsXVHACyl2eP6Cb15g+5Qgb6fk5kPB6WGpIwy3BfRN+jOOdfyNCIICP1tFlUVGq/cm1l7PIUHnvef6ZTes5nB+SsqjYfAdb7OzkI6jvxoPAUAQJKmZv7L7ELhWFhlX9DkK5n7ig0/E8STzg/tcUd7jdLlszJHvz/4H4BncMIMEYXsg3RZU/u+mtcwvqFKN5gQLR1hzjfMtAA2FiRuI32BG0vDOK3gmQ0Hdw3C7DyrO5hrwIGxgEMbHu+mrxibKcfY91USpih6OMEea98emU7tHgBq+gtQSwMEFAAAAAgA1IQvXZa3f8HsAQAAiQQAABgAAAB0dXJpbmdfbWFjaGluZS9lcnJvcnMucHmVU01rGzEQvftXDHuywU3vhhZcmkOgDqXxMWDGu7NegXYkRlJsU/LfOys5azspSbuH1dfovTdvRlVV3R5q8tE4DiBoAjWwPULsCKhPFqOTm6qqJpNJbTEEWCcxvFth3RmmWxEn0xFgtpiAfhr+DQNBudA6ARrigoJiBBSCgC1BdBA6t89jTA1xDFdMP8XtBPuHI0c8FKa35GfKJURBDmYQAlbPoUZmF2FL4FE0ryv078lbU2Ok9XircLylPXOs9w4kWQrQITeWsk0Be/1FhQLdhHDst85ecd3xE1rTrNwTfcyxzAxQOz0zWhNkSByS906i1qYjbKBXpF4Nu2JZmRDUndf5vOfZWuVLYtYA6EuEZhaAHaDP/myHJEfEwjdcbqiFzcawiZvNNJBt58WChQ4yP5nwsojkF2A4zrP6PJ3Bp69w75iKluHTHEmms5sRdjzJau/dZYGVX8U2Q3ctHrm6Cm2rUo0v8DtPnh+56Mk7eTZsqaoSQl6X2VddDuPzGW822vugcT9Mb+LQ7tRQ82/2ah1bs0uiWulAdcrqM7cdwGCvdgvp9Vf9+YuCS1LTf1IuLwl76p0cP/vSbUpTIC+Y6QT8t3a9Y5/i0voOtxQ/Zs7hoI/9ha44HfKDdykG05T3gta6varDE3Th/gNQSwMEFAAAAAgA1IQvXckt4fnLBQAA9xQAABkAAAB0dXJpbmdfbWFjaGluZS9tYWNoaW5lLnB5zVhLb+M2EL77V3B5khrFyF5dOGgQ+JAiWSwSo5cgEGibitnSlJeiNtlN/d93+JBISlS66KX1RRJnOJzHNw8aY7zeU9Qw8czhUbdyS1FdISVbtUdVLdG6lUBEd2S7Z4Ii+kq3rWK1QA09EKHYtpljjGezStYHVJZVq1pJyxKxw7GWChEhakX0hsbxbGvO6daszMlm2zFeE87JhtMC3Sgq9Ztl3xFFtpw0DW061n7JSZxTKWvZk7MZgt+N+Eo4292IY6uu+HFPNlStNF9hyHes0UavJREN08oEtHtqHXHLDkytXreU7uguoD8oekzRcqfOkciGyk6dz7J+luRQoPXq/u7m09Vt+bC+Wq8eHLMiR9qxruF9Npv91huYAc93KpZr2dJ8ZpbM6TeiqhdGlwa+FogJ5b6IouWGQuDoAr5ksEoq8Ktf3FOy6zm7/WbRMXZrUq813w6bmvvdL5IpRcVo/VB/dQf/kxn3rQA/t1xZO/aEK7pboE1d817ptvGSKyYIL40poWX02HhV61ZBtD1Z+1ZvkcrzmLVtLUB71cTusEwzp6AFvsO9VXJHK4A4E0yVpUWZUYLyqui/jjbciz7uPeUX/7rhRPxlDkdLhEvsKVo4iwzVHF8uAhYTpGNtYWt0BpYLT4dEql9oFzOwsUuoR5D2hP5Gn2pI5KV52F05Or80n4vIqLmzBXjdW0w2VgDRPGMSeIpAbMvIHGCNvtNbIvNgS/QdbxlYCswWaA1V2YCWI1YNPYNYg6A6WX9Q3lDzFh8hqRaWz/r424WJ4DNdbUqN+z50OAmARJgTcZmI9gSnPdzD3aMiEV/wRtKFgUs8s5Vu6ikI1fZ7Q0F2UlK8uer2x0L1TxIGnp8s19log/5hw4ha3RU61SH7G7ajSEFDc8rA04paIJyUc4ZwgfD8z5qJTNKjzKyw3LQ++w6aQ2OUUJ0yZ0Oej2TlMWpMUV+aej7XZR6iootJ4LbCZs3SJ1KBTOiWQRgHUpNJZH0bLQxRPZ2RsXwNtWHCafGDhXfFv5OtplhrUA5ONZXf4Kq3EVw+7JZ9AmoxmUk7SCh0IK+lqIXxYLmlnDeJHDH4j/tmmAOu90RBtaiEJqXYgVogVrgbgsAFhOu++M31LYMRo/mbt+KEg/gFjTkyNc7zmEEv9fSgDXd0jbK5Xs96dn+i6iebjt3V8LmnNPNnyOXMa1OExwQwZ1UoD8wf1wfrsPRYNXVCEQCjQIER42ya64mDekOLQCFL67QOlR6jI4IvEbvgBM9YQ9jRZQpbKYxMTovj0lVhOPfc9k1TI6xO1OxCXEtAb+NjT4gog/seXjqTztDHU1zV8kQ+ny3RG77FC3T+ESrdPbzo5wM8L06PgQ/15PaUrjcBl6CvKlU6nEZL9PFfJbcHOdweRJ+qsQMDrMTrQW4VqR0adtF6kGuJDQZecaADyEaECQxOMmkne2I8VUD1NPA2xWowHTszpSpQN7pGRQAqHpXQm7K44rRN5Pm57okyy/vC18fjDV9dX68+r3UzvF/9vrpe45Mt7nEIsI0nHpROy4plKwQkPw4MHAS2NysRWSuqcIoXYfDGZcIlrDxAP/4OnTkfQSL0VZEKbOj9VvzsRDcxe/1no53+6YLhb0JTMn6qU3r2WpT2btldzB8fu7Qs4rvRU2G2Tt8sEmA2Q0vn2emhMxjBR7U0mKUStCAGy+hrzByFYRl9pST3kQgntZjR457yd8czGDKnhysgBiegDxDzVP/5g/DWTSg49qmkX1omIT1f9lQgKD1Ssp3+M8cppPOjYs+tNP/P4P9557xxSpu+uSeN64UJBU6gZ9dlzeG/Jqb/Ctt2C1alOu5UY33ZM06NHyZHR+c92xKTTrOky2WQuRPXovRfTek7UYU1dzhFGNEnN15MTKkf5GniblRhmDrMDNHD862vo6fxnvhWxKBQ+O4DI/vYycvxUj70pCtD0yVC/xxTxkxxMk102Hm6SqJ7Tz77AVBLAwQUAAAACADUhC9dGanEH7YEAAB+DQAAGAAAAHR1cmluZ19tYWNoaW5lL3BhcnNlci5weaVX32/bNhB+91/Bcg+VMsVY+zAUKRI0Sw0sg5MGsVFsCAyBts4NO5kSSCpN5vl/3x1JWZTtdBnmB8s83R3vux8fac75jdAGNFtWmtl7YFYLZaSVlTq2Yl4C+zgZDznng8FSVyuW58vGNhrynMlVXWnLhFKVFWRggk4hrFiUwhgwrdJW5DVqYe9LOW/f3uDSv7BPtVRfWvmlBU0xBL9D0LrSW5/JgOHnY1OXciEsTLeBj0gtc28v1YMoZXGp6sael/W9mIPdf31VPUAkvdHVFy1WkydlxWMkvwVTNXoBY7mSdvS4ACigCO/TwWA6ur26vD4f55Pp+XQ0YacMo/4LlAGbrPmv5+Mpzxg/v7gY3bhft6PfRhdTvkHTwYdtghJvdDrVDaQDJ2IdtBMXiMF8wwk+tFtqEEVunlbzquyE37S0sCddIdJupeDR5jvOSqlwJZX916hCmnxIXduYE1bIhb2zWBi4Q7cZ+Z5lEYrZwNl8qHVVg7ZPblXA0gMziYFymbLjsy6B5Gfmd/KITVNazPC623fojH0fb4UIhJG3YRTfEGve4C7pZsffsKkRMCSRzy5D/8VxGjnGYVFRI/idqOSEt6bZy2ufyMTiZq4QDvv/SC8lxoOjoKmiuWpWcyBd18E5yQgCoBxnjEDj5kODs2TpHYLI2Ju0y7gzOI3NvXLCf+CkeffTDAugZZ104OWSITX4jtoK6bOolJWqga0Q82ANuo/89tyUoBKnk7JXp+xnhqjc8u7tjAT8+Iz3d9BCGjgwyElPy2WIjwnaOsrS5oTBYw0LCwV77YaZTf64+uXTmMoSfl19+jxi16Pfp37aX/Oe4y541ztZPKIZy7PedGZuKrNoGjETDt5uCiIvLhFvKBH0Inbn37wwHYfRW1ED894MWzXGsjmwCvUW90ILzIvmvfJQ/K7U2FFrPnbkRl8TvjkUxy7rvrQo0tv57db0/Upv3ne1GmfsNqOUTJ6rxp/wRE28V5MYDOkgjnjiet5qDQ+yaqhdI507NJsdwPrc8fRSzEVrH3MPTbVvlLV7YBow2wc8enRsHUFFXZYspcaiIgPhXgVW1o/3uoU2pOUmfS6Lu7AxEx26PrBD7f/93s9ivhr0Nw9setPyZRdHS6iuQdDLllMHvmSeSltL9jdxpj/Uj/xjJR79/sYdfqhxXTnGo4dXEWVZfYMWB+q11xN3OO1a7HO4I/vCz3bE+uGZUu9JIxVGoRbQijN/HkCJvRRE/uhGCgW9kkqUIe7WcTEM62O2cyMZhA7vsOKGbm5d5EIVjkz2XafsLE5Qd7q5Fn/+WtTvhSVv09/QzXD9zF6bcAt4j42ADinEdbf5hu/0BOLZKUwMqos1KNH44zG8YxKxWeAYPEH7jdznCYdn70IQ8n/oSrBnGUYTreL7Rm9QInmP4Hu+EH9wFQg4QNsqbWJyC/C+w8r7V+V9quoXsk171VgjC3D/Itpsi+Do5AA9/cjojBh+rSSdbLVO2vOrnx6Dd30okhB5mh4gpcALPv2BCcoKM7mdMfyD4W5WOKX0ZyNjR0eBKuhSzar5VzxCDly7kJawFcjE+Uh9gUicgFpUBf5ZOeWNXR6/471Q9niITPq7ImX9A1BLAwQUAAAACADUhC9dOzFxi7oCAAAlBwAAFgAAAHR1cmluZ19tYWNoaW5lL3RhcGUucHmNVT1v2zAQ3fUrrpqkRjaS1YCCLO3YoS26BIFASyebLUUKJNXG/fjvPZKSSNtNEQ4CTN7de/fuw3mefxqZNljBJPdqkh12YNmIoHHUaFBaZrmS2zzPs6zXaoCm6Sc7aWwa4MOotAUmpQpmZrbpmGWtYMagWYzWqwp6jqLLsuxhvcv8Fz4T8C4DOnvB5LcdGKuhhrzJ/WXTohBmBx1v7SOXtnLvT2TgAxYd9mwStulZa5U+1c6s8nnU75kwWGY+CplRDqMytuGS26YpDIq+hM09fFByxneH9yBQ+tet51PCmxruooE7mnGD8IWJCd9prXSRe1Mwp2GvBAyTsbBHwGciJU5AANAemaZfqPNyDeVBQn6Uzy9ix52eO/juIkOvNCx31XzHZeq15RYHU5SOdXgnrpH6n5D7g9d5QHtU3SqGq1hDSnJ5KFpB9XFl8dpX8La6KoVTnWm7IwKWbm69crFy7vgGqoGCFd67DvKt7y4d1fcGXQmDUJQNymlAzSwWjkB5rrMLuf2hKcvCw8PNRYQYXSO1p/QeseIaWedLWUEUlzLw7Cm33aV7qu0BbRHlTxoixp+pXQMs/LyC/+mykMSrOszLGxzMqzuMcGap67QxzrHSpEc1Jkk70jEY0ji96Pm4eLnRDJhz941ajajtaVVNKulZNC2tHhsHkXS7Ksg6iwEl0d7vLRO97TQKDAuCPk/w+1pyWlgp5QvFA6BzuiQx8HMSFQzs+QVahI2aZvGfxNzmiqi0XT8GgEJgb5NZb9VAPhadVBuv1caMtHBpWOmNEKQ1pd/OS6ygBixFDtqc9cFswc2FLkmetxUkIR0n2qT8cHQTH9yvy+Ns8nz7VS0a+ZlbUinPlpibd83kAYs09g3clYmAUumBCf4Tuyjhi6O6ql0+3j3FGEay0RxV0lvn/x5XwdzzRUH/AlBLAwQUAAAACADUhC9dUKLwHMMFAAAKFQAAGwAAAHR1cmluZ19tYWNoaW5lL3Zpc3VhbGl6ZS5wea1X227cNhB9368YqA+VElkxegHabRUUKFIgQBsEiNEX2xBoifKy4VIKyfXadfffO7zoQq3kS+t9sCVyOJw5c3iGiqLojMotE4SfNILfQSupokITzRoBG8pbKlUGZxsK9JaWOztMxTUTFKqGKhCNBrZtG6lBb5iCbVPtOM2iKFqtatlsoSjqnd5JWhSdHRG4yG6gVis/pu5U96jZlrqlFdGk5EQp3MdP9kPee7Yl5cYE4+c/adq+F3WTwtlOMnH9h5v2xi2RisrO9kwSoZiJY7VaVbQGTVpa7Jmomn3s/a5DPynsWaU3a2BCQw4/pvAqBclKHLlqGo5DvxGuaAInb0FpuV4B/uwSnNqS2/hb7yGxM5zW2k5Y59mGkgpOvP2bN/CNNSop52oNnCl9jj4vccH5pZ2pGwlt41LAiADzuaaxcZo616/9Zi4O81N32ysbZ7enyTmTuHHceUp6Y1YP7vMwzMFjH2JG2paKKq6jc0lvkDb08t5tdzh/041ExqkBDCjiBGjb2VxGw8Zm7oEd3ApnLimSS0CUZRlEmHCEj381TMR2RWJHcC7qSixJSQuOaSxWmCF91j2R4B/40BwXPqyw7pmEc8h8D57xBMydEeNkSClYEAeZ1pFJilT5vVluS1N4iGAvmaZ+wjxrKvq5aOJl29x0pubxYAJ2rwoPHy1Irak8DKsCOOvovqu2QiDU+vS76gAwGkQX659/OMD9zKHpOH64H/I8dBUokLVbogsUmhvW7FQx2MSz0IdQj2ANIe2IIHAU0BN6JBxsoEk0zm2AG7McAXJFMTLEaQb3Hrpl0DtfDusloJMJCILe6jEA85w8QqA/iYRrWi1hoL2wT0Do85pVgfEhTybkXo8k0xdn5KOVzbUk22ywV9k11QPcccCedBxKshooyOrx8Viqcx3tBCKJ7iqrgyE1sQLjAi7Xf9ipXzgampIgWlhqz2VvOJ7xdBiN2JK7vQZO/DL0NfsX/mRqRzj7m/qiV5STuzXUvCFGf06z74fW0ouSg09g3Ump2Q0Nm9LKO0L2tZygQg5NgfI6hWfq4WzjcwkFdJ1F3kFofEIoNIcL8clqy7SgF+JCTCXuo5eQgKL3jwlMMutrXscMMpkF2WWbmz/zDj5gXecDWTjkSSC+fXHkblqZ/m2hQv38q7HpbeGU25LDFy2YRpG44kR8LvwFY7DDaobmV3gQPjvSoqGi/iLyiLXd3Rij51ljS5RpY7wLG7+9uBnUsxIFpeH9Te9X97pgzJH+neXv+Lxg1hJBeWf30bwMd5DbkrYa3tu5d1I2MgzMUqPAak1P06Ra6VCLdAb3NAA3HYMXeEyCN3egVv1Yh03ewRIP9hiiQLbgnLmH2LBHGtGb7RleOg1UYSYWlNgl6xLt83I6YI/FmdzRJMWru+Y0jxw9wfMzStLAo4819/+HyQSIAlO4EOj9hmFmJvalttf9bGf0WI/vXfjFUUEgM/A2Hx2RI0ej8mXmPPamef+UAHyFF26Gl1iNH0clXh38d9HXCkqCRWYltl5qiDMXqInNV2Z+/7LZbk3ceQcYVq3d6Tg6f2eqd5mbMBD+HL2k8CX/smMasEUlmKFkbZxkvNlTGSez3jGCfgO8sn6J5oMwP8+1J3iRD3npWWi4coyIaS15UKT4+LDkx0PH6RkGZbsWOyo9PpTm9wClmf10fDalB6hq/Hx2a48tjkNF/OLx8Z8QFYfMZ93YIgG868ynNVKOGcKP/EylxcY2X7mhakN7nyTwIInNt3ymOHXVjE+zU99T7X0mGbW9OSV9Xvf7Xy3vib3uaU1usbu1GDieYPv96/WJCdyKc1r9BDX+N2BfkfIz6AYsGu6LNYtSqFEGc3WnsKAVykoy8boo0clg+TQpfTEZfUxCp7tOe9Oxx0EUX0oMnyiECyL4RAGcJjZ7nl5IAj3H8IIKr2FR5JIj8F9GiP6zCM0I0AxqR73Dd9LHmfOoEv0LUEsBAhQDFAAAAAgA1IQvXR1tJnv9AQAApAQAABoAAAAAAAAAAAAAAIABAAAAAHR1cmluZ19tYWNoaW5lL19faW5pdF9fLnB5UEsBAhQDFAAAAAgA1IQvXeJ8kwgwAAAAMAAAABoAAAAAAAAAAAAAAIABNQIAAHR1cmluZ19tYWNoaW5lL19fbWFpbl9fLnB5UEsBAhQDFAAAAAgA1IQvXSEnG+cnBAAAkA4AABkAAAAAAAAAAAAAAIABnQIAAHR1cmluZ19tYWNoaW5lL2NoZWNrZXIucHlQSwECFAMUAAAACADUhC9deO5+uwQFAAAlEAAAFQAAAAAAAAAAAAAAgAH7BgAAdHVyaW5nX21hY2hpbmUvY2xpLnB5UEsBAhQDFAAAAAgA1IQvXZa3f8HsAQAAiQQAABgAAAAAAAAAAAAAAIABMgwAAHR1cmluZ19tYWNoaW5lL2Vycm9ycy5weVBLAQIUAxQAAAAIANSEL13JLeH5ywUAAPcUAAAZAAAAAAAAAAAAAACAAVQOAAB0dXJpbmdfbWFjaGluZS9tYWNoaW5lLnB5UEsBAhQDFAAAAAgA1IQvXRmpxB+2BAAAfg0AABgAAAAAAAAAAAAAAIABVhQAAHR1cmluZ19tYWNoaW5lL3BhcnNlci5weVBLAQIUAxQAAAAIANSEL107MXGLugIAACUHAAAWAAAAAAAAAAAAAACAAUIZAAB0dXJpbmdfbWFjaGluZS90YXBlLnB5UEsBAhQDFAAAAAgA1IQvXVCi8BzDBQAAChUAABsAAAAAAAAAAAAAAIABMBwAAHR1cmluZ19tYWNoaW5lL3Zpc3VhbGl6ZS5weVBLBQYAAAAACQAJAHoCAAAsIgAAAAA='

try:
    import turing_machine
except ImportError:
    cur_dir = os.getcwd()
    engine_root = Path(cur_dir).parent / 'tools' / 'Turing Machine'
    source_dir = engine_root / 'src'
    EXAMPLES = {name: (engine_root / 'examples' / name).read_text(encoding='utf-8')
                for name in EMBEDDED_EXAMPLES}
    engine_origin = str(engine_root)
    sys.path.insert(0, str(source_dir))
except Exception as E:
    raise E

from turing_machine import TuringMachine, parse_program
from turing_machine.errors import MissingTransitionError, StepLimitExceededError

def make_machine(name, input_data):
    machine = TuringMachine(parse_program(EXAMPLES[name]))
    machine.reset(input_data)
    return machine

print('Engine:', engine_origin)
print('Programs:', ', '.join(EXAMPLES))


Engine: d:\Rep\MSAI_algorithms\MSAI_python\msai-python\tools\Turing Machine
Programs: binary_increment.tm, copy.tm, even_1.tm, invert_bits.tm, unary_addition.tm


### Interactive controls

The panel uses `ipywidgets`. If it is missing, uncomment and run the installation line below, then rerun the panel cells. For Colab, also uncomment the widget-manager lines if controls do not appear. A live Python kernel is required; static notebook previews cannot run buttons.


In [ ]:
# Run only if needed:
# %pip install ipywidgets
# For Google Colab, if needed:
# from google.colab import output
# output.enable_custom_widget_manager()


Choose a program, edit its input, and press **Reset / apply**. **Step** executes one rule; **Run N steps** executes a bounded batch. The highlighted rule is the *next* rule to execute. The tape window follows the head, with absolute cell indices.

Changing the selection loads that example's input and rules. Editing the input or rules takes effect only after **Reset / apply**. Every run has a 2,000-step budget; reaching it means **unknown**, not “loops forever”. Increase the budget in code only when needed.


In [8]:
from html import escape

DEFAULT_INPUTS = {
    'invert_bits.tm': '01001',
    'binary_increment.tm': '1011',
    'unary_addition.tm': '111+11',
    'even_1.tm': '1011',
    'copy.tm': '101',
}

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None

class MachinePanel:
    def __init__(self):
        self.machine = None
        self.previous = None
        self.failure = ''
        self.budget = 2000
        self.program = widgets.Dropdown(options=list(DEFAULT_INPUTS), description='Program:')
        self.input = widgets.Text(description='Input:')
        self.editor = widgets.Textarea(layout=widgets.Layout(width='100%', height='230px'))
        self.reset_button = widgets.Button(description='Reset / apply')
        self.step_button = widgets.Button(description='Step', button_style='primary')
        self.batch_button = widgets.Button(description='Run N steps')
        self.batch_size = widgets.BoundedIntText(value=20, min=1, max=500, description='N:')
        self.view = widgets.HTML()
        self.program.observe(self.select, names='value')
        self.reset_button.on_click(self.reset)
        self.step_button.on_click(lambda _: self.advance(1))
        self.batch_button.on_click(lambda _: self.advance(self.batch_size.value))
        self.controls = widgets.VBox([
            widgets.HBox([self.program, self.input]),
            widgets.HBox([self.reset_button, self.step_button, self.batch_button, self.batch_size]),
            widgets.HTML('<b>Transition editor — press Reset / apply after edits</b>'),
            self.editor, self.view,
        ])
        self.select()

    def select(self, change=None):
        name = self.program.value
        self.input.value = DEFAULT_INPUTS[name]
        self.editor.value = EXAMPLES[name]
        self.reset()

    def reset(self, button=None):
        self.machine = None
        self.previous = None
        self.failure = ''
        try:
            self.loaded_source = self.editor.value
            self.machine = TuringMachine(parse_program(self.loaded_source))
            self.machine.reset(self.input.value)
        except Exception as exc:
            self.failure = f'{type(exc).__name__}: {exc}'
        self.render()

    def advance(self, count):
        if self.machine is None or self.failure:
            return
        for _ in range(min(count, 500)):
            if self.machine.halted or self.machine.steps >= self.budget:
                break
            try:
                self.previous = self.machine.step()
            except Exception as exc:
                self.failure = f'{type(exc).__name__}: {exc}'
                break
        self.render()

    def render(self):
        m = self.machine
        disabled = (m is None or bool(self.failure) or m.halted or m.steps >= self.budget)
        self.step_button.disabled = disabled
        self.batch_button.disabled = disabled
        if m is None:
            self.view.value = '<pre>' + escape(self.failure) + '</pre>'
            return
        status = ('ERROR: ' + self.failure if self.failure else
                  m.result().status if m.halted else
                  'UNKNOWN — step budget reached' if m.steps >= self.budget else 'running')
        positions = range(m.head - 8, m.head + 9)
        indices = ''.join(f'<th style="padding:6px">{p}</th>' for p in positions)
        symbols = ''.join(
            '<td style="text-align:center;padding:8px;border:1px solid #888;'
            + ('background:#ffe082;color:#111;font-weight:bold' if p == m.head else '')
            + '">' + escape(m.tape.read(p)) + (' ↑' if p == m.head else '') + '</td>'
            for p in positions)
        next_rule = None if m.halted else m.program.transitions.get((m.state, m.tape.read(m.head)))
        rule_lines = []
        for line, text in enumerate(self.loaded_source.splitlines(), 1):
            style = 'background:#ffe082;color:#111' if next_rule and line == next_rule.line else ''
            rule_lines.append(f'<div style="{style}">{line:02d}  {escape(text)}</div>')
        previous = 'none'
        if self.previous:
            p = self.previous
            previous = f'{p.state_before} {p.read_symbol} -> {p.written_symbol} {p.move} {p.state_after}'
        next_text = ('terminal state' if m.halted else
                     f'{next_rule.state} {next_rule.read_symbol} -> {next_rule.write_symbol} {next_rule.move} {next_rule.next_state}'
                     if next_rule else 'undefined transition')
        self.view.value = (
            f'<p><b>{escape(status)}</b> · step {m.steps} / {self.budget} · state {escape(m.state)} · head {m.head}</p>'
            f'<table><tr>{indices}</tr><tr>{symbols}</tr></table>'
            f'<p>Previous: <code>{escape(previous)}</code><br>Next: <code>{escape(next_text)}</code></p>'
            f'<p>Nonblank span (outer blanks omitted): <code>{escape(repr(m.tape.normalized()))}</code></p>'
            '<pre style="max-height:300px;overflow:auto">' + ''.join(rule_lines) + '</pre>')

if widgets is not None:
    panel = MachinePanel()
    display(panel.controls)
else:
    print('ipywidgets is missing. Install it above, or use the manual controls below.')


### Manual controls / direct Python API

These cells work even without widgets. Rerun the second cell for one more step; rerun the first to reset. The brackets mark the head. Use this API when explaining what the buttons do.


In [ ]:
from turing_machine.visualize import trace_line
manual = make_machine('binary_increment.tm', '1011')
print(trace_line(manual, None))


In [ ]:
if not manual.halted and manual.steps < 2000:
    info = manual.step()
    print(trace_line(manual, info))
else:
    print('Stopped:', manual.result())


## 4. Walk through the five example programs

For each example: predict the output → select it in the panel → step through the interesting transition → explain the invariant → check a boundary input.

### A. `invert_bits.tm`: one left-to-right pass

For a binary word `w`, produce its bitwise complement. Invariant: cells left of the head have been inverted; unvisited input cells are unchanged. The first blank stops the machine.

Try `01001`, `0`, and the empty input. For length `n`, how many transitions execute, including the final blank rule?


In [ ]:
print(EXAMPLES['invert_bits.tm'])
for data, expected in [('01001', '10110'), ('0', '1'), ('', '')]:
    result = make_machine('invert_bits.tm', data).run(max_steps=2000)
    assert result.output == expected and result.steps == len(data) + 1
    print(repr(data), '->', repr(result.output), 'steps:', result.steps)


### B. `binary_increment.tm`: carrying is a state

The first phase finds the right edge; `q_carry` moves left through trailing ones, replacing them with zeros. A zero absorbs the carry. If all bits were ones, a leading one is written at a negative tape index.

**Predict:** `1011 + 1`, `111 + 1`, and `0 + 1`. On empty input this particular program writes `1`; treat this as its convention, not a universal binary-number syntax rule.


In [ ]:
print(EXAMPLES['binary_increment.tm'])
for data, expected in [('1011', '1100'), ('111', '1000'), ('0', '1'), ('', '1')]:
    result = make_machine('binary_increment.tm', data).run(max_steps=2000)
    assert result.output == expected
    print(repr(data), '->', repr(result.output), 'leftmost nonblank index:', result.tape_start)


### C. `copy.tm`: mark, carry, return, restore

Output is `w@w`, not just `ww`. First write the separator `@` and return to the beginning. In `q2`, mark the current source bit as `X`; remember its value in state `r0` or `r1`; scan to the end and append that bit. Return in `l0` or `l1`, restore `X` to the original bit, and move to the next source position. Reaching `@` in `q2` means all bits were copied.

**Invariant:** at the start of each `q2` iteration, the copied suffix equals the already processed prefix. The temporary `X` is restored before the next iteration.

Trace `10` or `101` in the panel. Why are two outbound states needed? Why must the original bit be restored? The empty word produces `@`.


In [9]:
print(EXAMPLES['copy.tm'])
for data in ['10', '101', '']:
    result = make_machine('copy.tm', data).run(max_steps=2000)
    assert result.output == data + '@' + data
    print(repr(data), '->', repr(result.output), 'steps:', result.steps)


q0 0 -> 0 R q0
q0 1 -> 1 R q0
q0 _ -> @ L q1
q1 0 -> 0 L q1
q1 1 -> 1 L q1
q1 _ -> _ R q2
q2 0 -> X R r0
q2 1 -> X R r1

r0 0 -> 0 R r0
r0 1 -> 1 R r0
r0 @ -> @ R r0
r1 0 -> 0 R r1
r1 1 -> 1 R r1
r1 @ -> @ R r1

r0 _ -> 0 L l0
r1 _ -> 1 L l1

l0 0 -> 0 L l0
l0 1 -> 1 L l0
l0 @ -> @ L l0
l1 0 -> 0 L l1
l1 1 -> 1 L l1
l1 @ -> @ L l1

l0 X -> 0 R q2
l1 X -> 1 R q2

q2 @ -> @ S HALT
'10' -> '10@10' steps: 21
'101' -> '101@101' steps: 36
'' -> '@' steps: 3


**Short experiment:** compare copying inputs of lengths 2, 4, 8, and 16. Explain the growth using the repeated trips across the tape. Timing is unnecessary; count transitions. For this implementation, the number of full trips grows with input length and each trip crosses a span proportional to that length: quadratic growth.


In [10]:
for n in [2, 4, 8, 16]:
    result = make_machine('copy.tm', '1' * n).run(max_steps=10000)
    print('length:', n, 'steps:', result.steps)


length: 2 steps: 21
length: 4 steps: 55
length: 8 steps: 171
length: 16 steps: 595


## 5. The halting problem

Given an arbitrary program `P` and input `x`, can one algorithm always finish and correctly decide whether `P(x)` eventually halts?

**The answer is no.** This is about a universal, always-correct, always-terminating decision procedure. It does not prevent us from proving termination or nontermination for particular programs.

### A timeout is not a verdict

A bounded simulator can answer “halted within this budget” or “unknown after this budget”. An undefined transition is a separate error under our emulator's rules.


In [ ]:
def bounded_run(source, data='', budget=100):
    machine = TuringMachine(parse_program(source))
    machine.reset(data)
    try:
        machine.run(max_steps=budget)
        return {'verdict': 'HALTED', 'steps': machine.steps, 'output': machine.tape.normalized()}
    except StepLimitExceededError:
        return {'verdict': 'UNKNOWN', 'steps': machine.steps}
    except MissingTransitionError as exc:
        return {'verdict': 'UNDEFINED TRANSITION', 'steps': machine.steps, 'detail': str(exc)}

stationary_loop = 'q0 _ -> _ S q0'
drifting_loop = 'q0 _ -> _ R q0'
for label, source, data, budget in [
    ('Increment, small budget', EXAMPLES['binary_increment.tm'], '111', 2),
    ('Increment, sufficient budget', EXAMPLES['binary_increment.tm'], '111', 100),
    ('Stationary loop', stationary_loop, '', 100),
    ('Drifting loop', drifting_loop, '', 100),
    ('Missing rule', 'q0 0 -> 0 S HALT', '', 100),
]:
    print(label, bounded_run(source, data, budget))


Both loop examples above can be proved nonterminating directly from their rules. The bounded simulator alone did not prove that. In particular, the same `UNKNOWN` label was also returned for an increment that eventually halts.

### Why a universal decider is impossible

Assume a total algorithm `H(program, input)` exists: it always finishes, returning `True` exactly when the supplied computation halts. Programs can be encoded as data, so a program description can also be used as input.

Construct `D` using this hypothetical algorithm:

```text
D(program_text):
    if H(program_text, program_text):
        loop forever
    else:
        halt
```

Now ask what happens to `D(description_of_D)`.

| H's prediction | What D does by construction | Contradiction |
| --- | --- | --- |
| It halts | Loops forever | Prediction is wrong |
| It does not halt | Halts | Prediction is wrong |

Both possibilities contradict the assumed correctness of `H`. Therefore such a total universal decider cannot exist. The pseudocode is an argument under an impossible assumption; we do not implement or execute `H` or an infinite Python loop here.

Simulation can recognize halting by eventually reaching it; on nonhalting inputs it may wait forever. This is why the halting set is recognizable but not decidable.


### Optional: can remembering configurations detect loops?

For a deterministic machine, repeating the **entire configuration** proves an infinite cycle: the next steps must repeat too. A repeated state alone is insufficient, because the head or tape may have changed.

The detector below compares exact state, head index, and every nonblank cell. It remains bounded and may answer `UNKNOWN`. It is a useful partial method, not a universal decider.


In [ ]:
def detect_repeated_configuration(source, data='', budget=200):
    machine = TuringMachine(parse_program(source))
    machine.reset(data)
    seen = set()
    for step in range(budget + 1):
        if machine.halted:
            return 'HALTED'
        configuration = (machine.state, machine.head,
                         tuple(sorted(machine.tape.snapshot().items())))
        if configuration in seen:
            return 'PROVED LOOP: repeated full configuration'
        seen.add(configuration)
        if step == budget:
            return 'UNKNOWN'
        try:
            machine.step()
        except MissingTransitionError:
            return 'UNDEFINED TRANSITION'

assert detect_repeated_configuration(stationary_loop).startswith('PROVED LOOP')
assert detect_repeated_configuration(drifting_loop) == 'UNKNOWN'
assert detect_repeated_configuration(EXAMPLES['invert_bits.tm'], '010') == 'HALTED'
print('Stationary:', detect_repeated_configuration(stationary_loop))
print('Drifting:  ', detect_repeated_configuration(drifting_loop))


The drifting machine never repeats its absolute head position, so this exact-configuration detector misses a loop that we can prove by inspection. More sophisticated methods can prove more cases, but none can decide every unrestricted program/input pair.

**Connection to finite automata:** a system with finitely many possible configurations must halt or repeat a configuration. A Turing machine has unbounded tape and head positions, so there is no universal finite bound to apply this argument. A fixed tape bound creates a different, restricted problem.

## 6. Exit questions

1. Why can `a == b` be true while `a is b` is false? What does `sys.intern` explicitly change?
2. What three components determine the next step of a deterministic Turing machine?
3. What does `even_1.tm` output for the empty input? Does it enter `ACCEPT`?
4. A machine ran for a million steps without halting. What can you conclude?
5. Why is a repeated state not sufficient to prove a loop?
6. Which assumption about `H` is contradicted by the diagonal construction?

**Try after class:** edit one rule in the panel, predict the effect, then test it. Restore the original via the program selector. Can you make the bit inverter continue moving right on blanks? Explain its behavior using the rules, not just a timeout.

## References and local materials

- [Python: sys.intern](https://docs.python.org/3/library/sys.html#sys.intern).
- [CPython integer-object cache](https://docs.python.org/3/c-api/long.html#c.PyLong_FromLong).
- [Week 00 homework](../week00_python_survival_kit/W00.tasks.ipynb).
- [Turing machine example programs](../tools/Turing%20Machine/examples/) and [engine](../tools/Turing%20Machine/src/turing_machine/machine.py).

Local links work in a full checkout. The machine examples and runtime are also embedded in this notebook for standalone use.
